### Hands-on Task 1: Extend the Basic Pipeline (No LLM)

#### Objective 

Enhance an existing pipeline by adding one more processing step. 

#### Task 

Introduce a new node called word_count with the following functionality: 

Count the number of words in the input text	 

Store the result in a new state field: 

word_count: int 

Update the pipeline flow as follows: 

START → count → word_count → summarise → END 

Modify the make_summary node so that the final output includes both: 

Character count 

Word count 

Expected Output 


Summary: 'LangGraph makes stateful AI workflows easy!' — 43 characters, 6 words. 

Skills Tested 

Extending TypedDict state 

Adding and connecting new nodes 

Passing state across nodes 

Using add_edge 

In [4]:
%pip install langgraph --break-system-packages

Defaulting to user installation because normal site-packages is not writeable
  Using cached websockets-15.0.1-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 246.9/246.9 kB 5.2 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.2/56.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 kB 1.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.5/160.5 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.6/212.6 kB 8.9 MB/s eta 0:00:00
Using cached websockets-15.0.1-cp312-cp312-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl (182 kB)
  Attempting uninstall: websockets
    Found existing installation: websockets 16.0
    Uninstalling websockets-16.0:
      Successfully uninstalled websockets-16.0
Note: you may need to restart the kernel to use updated packages.


In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# ==========================================
# 1. Extend the TypedDict State
# ==========================================
class PipelineState(TypedDict):
    text: str
    char_count: int
    word_count: int  # <-- Added new state field
    summary: str

# ==========================================
# 2. Define/Update the Nodes
# ==========================================
def count_chars(state: PipelineState) -> dict:
    """Existing node: counts characters."""
    return {"char_count": len(state["text"])}

def word_count_node(state: PipelineState) -> dict:
    """New node: counts words in the input text."""
    words = state["text"].split()
    return {"word_count": len(words)}

def make_summary(state: PipelineState) -> dict:
    """Updated node: includes both character and word counts."""
    text = state["text"]
    char_count = state["char_count"]
    word_count = state["word_count"]
    
    summary_str = f"Summary: '{text}' — {char_count} characters, {word_count} words."
    return {"summary": summary_str}

# ==========================================
# 3. Build and Connect the Graph Flow
# ==========================================
# START → count → word_count → summarise → END

workflow = StateGraph(PipelineState)

# Add nodes
workflow.add_node("count", count_chars)
workflow.add_node("word_count", word_count_node)
workflow.add_node("summarise", make_summary)

# Set up the updated execution flow
workflow.add_edge(START, "count")
workflow.add_edge("count", "word_count")
workflow.add_edge("word_count", "summarise")
workflow.add_edge("summarise", END)

# Compile the graph
app = workflow.compile()

# ==========================================
# 4. Invoke and Test
# ==========================================
initial_state = {"text": "LangGraph makes stateful AI workflows easy!"}
final_output = app.invoke(initial_state)

print(final_output["summary"])

Summary: 'LangGraph makes stateful AI workflows easy!' — 43 characters, 6 words.
